In [1]:
!pip install -q \
    pymupdf \
    sentence-transformers \
    faiss-cpu \
    google-genai \
    gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 43.2 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import getpass
from pathlib import Path
from typing import List, Dict, Any, Tuple

import fitz
import faiss
import numpy as np
import gradio as gr

from sentence_transformers import SentenceTransformer
from google import genai

GEMINI_API_KEY = getpass.getpass(
    "Enter your Gemini API key. It will not be displayed: "
)

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client configured successfully.")

Enter your Gemini API key. It will not be displayed: ··········
Gemini client configured successfully.


In [3]:
from google.colab import files

uploaded_files = files.upload()

pdf_paths = [
    filename
    for filename in uploaded_files.keys()
    if filename.lower().endswith(".pdf")
]

if not pdf_paths:
    raise ValueError("No PDF was uploaded. Please upload at least one PDF.")

print(f"Uploaded {len(pdf_paths)} PDF file(s):")

for pdf_path in pdf_paths:
    print("-", pdf_path)

Saving LangChain.pdf to LangChain.pdf
Uploaded 1 PDF file(s):
- LangChain.pdf


In [4]:
def clean_text(text: str) -> str:
    """
    Clean common whitespace problems while preserving readable text.
    """
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract text page-by-page from a PDF.

    Returns:
        A list containing:
        - document name
        - page number
        - extracted text
    """
    document = fitz.open(pdf_path)
    pages = []

    for page_index in range(len(document)):
        page = document.load_page(page_index)
        page_text = clean_text(page.get_text("text"))

        if page_text:
            pages.append(
                {
                    "document": Path(pdf_path).name,
                    "page": page_index + 1,
                    "text": page_text,
                }
            )

    document.close()

    return pages

In [5]:
all_pages = []

for pdf_path in pdf_paths:
    extracted_pages = extract_pdf_pages(pdf_path)
    all_pages.extend(extracted_pages)

print(f"Successfully extracted {len(all_pages)} pages.")

if all_pages:
    print("\nSample page information:")
    print("Document:", all_pages[0]["document"])
    print("Page:", all_pages[0]["page"])
    print("Text preview:")
    print(all_pages[0]["text"][:800])

Successfully extracted 14 pages.

Sample page information:
Document: LangChain.pdf
Page: 1
Text preview:
LangChain
Vasilios Mavroudis
Alan Turing Institute
vmavroudis@turing.ac.uk
Abstract. LangChain is a rapidly emerging framework that offers a ver-
satile and modular approach to developing applications powered by large
language models (LLMs). By leveraging LangChain, developers can sim-
plify complex stages of the application lifecycle—such as development,
productionization, and deployment—making it easier to build scalable,
stateful, and contextually aware applications. It provides tools for han-
dling chat models, integrating retrieval-augmented generation (RAG),
and offering secure API interactions. With LangChain, rapid deployment
of sophisticated LLM solutions across diverse domains becomes feasible.
However, despite its strengths, LangChain’s emphasis on modularity and
integration int


In [6]:
def create_chunks(
    pages: List[Dict[str, Any]],
    chunk_size: int = 180,
    chunk_overlap: int = 40,
) -> List[Dict[str, Any]]:
    """
    Split each PDF page into overlapping word chunks.

    Args:
        pages:
            Extracted PDF pages.

        chunk_size:
            Maximum number of words in one chunk.

        chunk_overlap:
            Number of words shared with the next chunk.

    Returns:
        List of chunks with document and page metadata.
    """
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size.")

    chunks = []
    chunk_id = 0
    step_size = chunk_size - chunk_overlap

    for page in pages:
        words = page["text"].split()

        for start_index in range(0, len(words), step_size):
            end_index = start_index + chunk_size
            chunk_words = words[start_index:end_index]

            # Ignore extremely small fragments.
            if len(chunk_words) < 20:
                continue

            chunk_text = " ".join(chunk_words)

            chunks.append(
                {
                    "chunk_id": chunk_id,
                    "document": page["document"],
                    "page": page["page"],
                    "text": chunk_text,
                }
            )

            chunk_id += 1

    return chunks

In [7]:
chunks = create_chunks(
    pages=all_pages,
    chunk_size=180,
    chunk_overlap=40,
)

print(f"Created {len(chunks)} chunks.")

if chunks:
    print("\nExample chunk:")
    print("Chunk ID:", chunks[0]["chunk_id"])
    print("Document:", chunks[0]["document"])
    print("Page:", chunks[0]["page"])
    print("Text:")
    print(chunks[0]["text"])

Created 41 chunks.

Example chunk:
Chunk ID: 0
Document: LangChain.pdf
Page: 1
Text:
LangChain Vasilios Mavroudis Alan Turing Institute vmavroudis@turing.ac.uk Abstract. LangChain is a rapidly emerging framework that offers a ver- satile and modular approach to developing applications powered by large language models (LLMs). By leveraging LangChain, developers can sim- plify complex stages of the application lifecycle—such as development, productionization, and deployment—making it easier to build scalable, stateful, and contextually aware applications. It provides tools for han- dling chat models, integrating retrieval-augmented generation (RAG), and offering secure API interactions. With LangChain, rapid deployment of sophisticated LLM solutions across diverse domains becomes feasible. However, despite its strengths, LangChain’s emphasis on modularity and integration introduces complexities and potential security concerns that warrant critical examination. This paper provides an in-d

In [8]:

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [9]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

chunk_embeddings = np.asarray(
    chunk_embeddings,
    dtype="float32",
)

print("Embedding matrix shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding matrix shape: (41, 384)


In [10]:
embedding_dimension = chunk_embeddings.shape[1]

vector_index = faiss.IndexFlatIP(embedding_dimension)
vector_index.add(chunk_embeddings)

print("Number of vectors stored:", vector_index.ntotal)

Number of vectors stored: 41


In [11]:
def retrieve_chunks(
    question: str,
    top_k: int = 5,
) -> List[Dict[str, Any]]:
    """
    Retrieve the most relevant PDF chunks for a question.
    """
    question = question.strip()

    if not question:
        return []

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    question_embedding = np.asarray(
        question_embedding,
        dtype="float32",
    )

    number_to_retrieve = min(top_k, len(chunks))

    scores, indices = vector_index.search(
        question_embedding,
        number_to_retrieve,
    )

    results = []

    for rank, (chunk_index, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1,
    ):
        if chunk_index == -1:
            continue

        chunk = chunks[int(chunk_index)].copy()
        chunk["similarity_score"] = float(score)
        chunk["rank"] = rank

        results.append(chunk)

    return results

In [12]:
test_question = "What are the main topics discussed in this document?"

retrieved_results = retrieve_chunks(
    question=test_question,
    top_k=3,
)

for result in retrieved_results:
    print("=" * 80)
    print(
        f"Rank: {result['rank']} | "
        f"Document: {result['document']} | "
        f"Page: {result['page']} | "
        f"Score: {result['similarity_score']:.3f}"
    )
    print()
    print(result["text"][:700])

Rank: 1 | Document: LangChain.pdf | Page: 3 | Score: 0.257

then generates an accurate and contextually enriched answer. This ar- chitecture enhances the model’s ability to produce factually grounded responses by incorporating relevant knowledge from the vector store. The rest of this section provides an overview of LangChain’s primary com- ponents, followed by a brief introduction to its advanced modules–LangSmith, LangGraph and LangServe–which are further discussed in Sections 2, 3, and 4 respectively: LLM Interface: Provides APIs for connecting and querying various large lan- guage models, such as OpenAI’s GPT [1], Google’s Gemini [14], and Llama [16], to facilitate seamless application integration. Prompt Templates: Structured templates that stan
Rank: 2 | Document: LangChain.pdf | Page: 2 | Score: 0.240

integrations and third-party providers necessitates a careful examination of security practices to mitigate risks associated with data exposure and dependency vulnerabilities. Thi

In [13]:
def build_context(
    retrieved_chunks: List[Dict[str, Any]]
) -> Tuple[str, List[str]]:
    """
    Format retrieved chunks as numbered sources.
    """
    context_sections = []
    source_labels = []

    for source_number, chunk in enumerate(
        retrieved_chunks,
        start=1,
    ):
        source_label = (
            f"Source {source_number}: "
            f"{chunk['document']}, page {chunk['page']}"
        )

        source_labels.append(source_label)

        context_sections.append(
            f"""
[{source_label}]
{chunk["text"]}
""".strip()
        )

    context = "\n\n".join(context_sections)

    return context, source_labels


In [14]:
def create_rag_prompt(
    question: str,
    context: str,
) -> str:
    """
    Create strict instructions for grounded PDF question answering.
    """
    return f"""
You are a document question-answering assistant.

Answer the user's question using only the provided document context.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts.
3. If the context does not contain the answer, say:
   "I could not find enough information in the uploaded documents."
4. Cite supporting sources using the format:
   [Source 1], [Source 2]
5. Give a clear and concise answer.
6. When appropriate, use short bullet points.
7. Do not claim that a source supports something unless it actually does.

DOCUMENT CONTEXT:

{context}

USER QUESTION:

{question}

ANSWER:
""".strip()

In [15]:
def answer_question(
    question: str,
    top_k: int = 5,
) -> Dict[str, Any]:
    """
    Complete RAG pipeline:
    question → retrieval → prompt → Gemini answer
    """
    question = question.strip()

    if not question:
        return {
            "answer": "Please enter a question.",
            "sources": [],
            "retrieved_chunks": [],
        }

    retrieved = retrieve_chunks(
        question=question,
        top_k=top_k,
    )

    if not retrieved:
        return {
            "answer": (
                "I could not retrieve relevant information "
                "from the uploaded documents."
            ),
            "sources": [],
            "retrieved_chunks": [],
        }

    context, source_labels = build_context(retrieved)

    prompt = create_rag_prompt(
        question=question,
        context=context,
    )

    # Fixed: Using the full model path to resolve 404 error
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt,
    )

    answer = response.text or "No answer was generated."

    return {
        "answer": answer,
        "sources": source_labels,
        "retrieved_chunks": retrieved,
    }

In [16]:

question = input("Ask a question about the uploaded PDFs: ")

result = answer_question(
    question=question,
    top_k=5,
)

print("\nANSWER")
print("-" * 80)
print(result["answer"])

print("\nRETRIEVED SOURCES")
print("-" * 80)

for source in result["sources"]:
    print(source)

Ask a question about the uploaded PDFs: what is langchain?

ANSWER
--------------------------------------------------------------------------------
Based on the provided documents, **LangChain** is a versatile framework with a modular architecture designed to simplify the lifecycle of applications powered by large language models (LLMs) [Source 2], [Source 3]. 

Key aspects of LangChain include:
* **Lifecycle Management:** It supports LLM-powered applications from initial development through to deployment and monitoring [Source 2].
* **Modular Architecture:** Its modularity allows developers to configure, extend, and deploy applications tailored to specific needs [Source 2].
* **Flexible Foundation:** It provides a flexible foundation for building scalable, secure, and multi-functional applications [Source 4].
* **Bridge to Practical AI:** It bridges the gap between the power of large language models and practical application development across multiple fields [Source 5].

RETRIEVED SO

In [17]:
for retrieved_chunk in result["retrieved_chunks"]:
    print("=" * 100)

    print(
        f"Rank: {retrieved_chunk['rank']} | "
        f"Document: {retrieved_chunk['document']} | "
        f"Page: {retrieved_chunk['page']} | "
        f"Similarity: {retrieved_chunk['similarity_score']:.3f}"
    )

    print()
    print(retrieved_chunk["text"])
    print()

Rank: 1 | Document: LangChain.pdf | Page: 14 | Similarity: 0.621

14 Vasilios Mavroudis References 1. Josh Achiam, Steven Adler, Sandhini Agarwal, Lama Ahmad, Ilge Akkaya, Flo- rencia Leoni Aleman, Diogo Almeida, Janko Altenschmidt, Sam Altman, Shyamal Anadkat, et al. GPT-4 Technical Report. arXiv preprint arXiv:2303.08774, 2023. 2. Harrison Chase. LangChain, Oct 2022. Available at https://github.com/ langchain-ai/langchain. 3. LangChain, Inc. LangChain Documentation: Integration Providers. LangChain, Inc., San Francisco, CA, 2024. Available at https://python.langchain.com/docs/ integrations/providers/. 4. LangChain, Inc. LangChain Documentation: Key Concepts. LangChain, Inc., San Francisco, CA, 2024. Available at https://python.langchain.com/docs/ concepts/. 5. LangChain, Inc. LangChain Documentation: LangServe. LangChain, Inc., San Francisco, CA, 2024. Available at https://python.langchain.com/docs/ langserve/. 6. LangChain, Inc. LangChain Documentation: Security Best Practices. Lang

In [18]:

def chatbot_response(
    question: str,
    number_of_sources: int,
) -> Tuple[str, str]:
    """
    Function used by the Gradio interface.
    """
    result = answer_question(
        question=question,
        top_k=int(number_of_sources),
    )

    source_text = "\n".join(
        f"- {source}"
        for source in result["sources"]
    )

    if not source_text:
        source_text = "No sources retrieved."

    retrieval_details = []

    for chunk in result["retrieved_chunks"]:
        retrieval_details.append(
            (
                f"### Rank {chunk['rank']}\n"
                f"**Document:** {chunk['document']}  \n"
                f"**Page:** {chunk['page']}  \n"
                f"**Similarity:** "
                f"{chunk['similarity_score']:.3f}\n\n"
                f"{chunk['text']}"
            )
        )

    retrieval_markdown = "\n\n---\n\n".join(retrieval_details)

    final_answer = (
        f"{result['answer']}\n\n"
        f"### Retrieved sources\n"
        f"{source_text}"
    )

    return final_answer, retrieval_markdown












with gr.Blocks(title="PDF Knowledge Assistant") as demo:
    gr.Markdown(
        """
        # PDF Knowledge Assistant

        Ask questions about the PDFs uploaded in this Colab notebook.

        The assistant retrieves relevant passages before generating an answer.
        """
    )

    question_box = gr.Textbox(
        label="Question",
        placeholder="Example: What methodology does the document describe?",
        lines=3,
    )

    top_k_slider = gr.Slider(
        minimum=1,
        maximum=10,
        value=5,
        step=1,
        label="Number of chunks to retrieve",
    )

    ask_button = gr.Button("Ask the PDFs")

    answer_output = gr.Markdown(
        label="Answer"
    )

    with gr.Accordion(
        "View retrieved PDF chunks",
        open=False,
    ):
        retrieval_output = gr.Markdown()

    ask_button.click(
        fn=chatbot_response,
        inputs=[
            question_box,
            top_k_slider,
        ],
        outputs=[
            answer_output,
            retrieval_output,
        ],
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5fcc6095cc15c0145a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
